# AI Gym Coach — Cloud Serving & Gateway Pipeline
This notebook sets up the complete inference pipeline on a cloud GPU instance (Kaggle T4/P100 or Colab):
1. Installs pinned dependencies & sanitizes dynamic C-extensions.
2. Pulls the latest repo state and fetches the fine-tuned LoRA adapter from Hugging Face (`ahmedhassanM/qwen2.5-7b-gym-coach-lora`).
3. Spawns `vLLM` on port `8001` with 4-bit `BitsAndBytes` quantization & bound LoRA weights.
4. Launches the `FastAPI` 3-class intent triage gateway on port `8000`.
5. Exposes a secure public HTTPS endpoint via `ngrok` with real-time streaming server logs.

In [ ]:
# CELL 1: DEPENDENCIES & CLEANUP
import os
import sys
import subprocess

print("[1/5] Installing dependencies and sanitizing container environment...")

os.environ["ANONYMIZED_TELEMETRY"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

cmd = [
    sys.executable, "-m", "pip", "install", "-q",
    "protobuf<6.0.0,>=5.26.1",
    "pyngrok>=7.0.0",
    "fastapi",
    "uvicorn",
    "httpx",
    "chromadb",
    "sentence-transformers",
    "bitsandbytes",
    "rapidfuzz",
    "vllm",
    "psutil",
    "huggingface_hub"
]
subprocess.run(cmd, check=True)

# Remove torchcodec to prevent vLLM C-extension dynamic linking errors
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchcodec"], stderr=subprocess.DEVNULL)

print("✓ Core dependencies installed and C-extensions sanitized.")

In [ ]:
# CELL 2: GIT SYNC, ASSET VERIFICATION & HUGGING FACE LORA FETCH
import os
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from huggingface_hub import snapshot_download

REPO_URL = "https://github.com/AhmedH32/ai-gym-coach.git"
HF_LORA_REPO = "ahmedhassanM/qwen2.5-7b-gym-coach-lora"
PROJECT_ROOT = Path("/kaggle/working/ai-gym-coach").resolve()

# 1. Clone fresh or sync origin/main
if not PROJECT_ROOT.exists():
    print(f"[2/5] Cloning repository into {PROJECT_ROOT}...")
    !git clone {REPO_URL} /kaggle/working/ai-gym-coach
else:
    print(f"[2/5] Repository exists. Resetting hard to origin/main...")

%cd /kaggle/working/ai-gym-coach
!git fetch --all
!git checkout main
!git reset --hard origin/main
!git pull origin main

# 2. Assert core source files exist
assert (PROJECT_ROOT / "backend" / "agent_core" / "orchestrator.py").exists(), "FATAL: orchestrator.py not found."
assert (PROJECT_ROOT / "backend" / "server.py").exists(), "FATAL: server.py not found."

# 3. Verify exercises.json presence
catalog_candidates = [
    PROJECT_ROOT / "raw_data" / "exercises.json",
    PROJECT_ROOT / "client" / "src" / "assets" / "data" / "exercises.json"
]
catalog_path = next((p for p in catalog_candidates if p.exists()), None)
if not catalog_path:
    raise FileNotFoundError("FATAL: exercises.json not found.")

with open(catalog_path, "r", encoding="utf-8") as f:
    catalog_data = json.load(f)
    count = len(catalog_data if isinstance(catalog_data, list) else catalog_data.get("exercises", []))
print(f"✓ Catalog verified ({count} movements indexed).")

# 4. Fetch LoRA Adapter directly from Hugging Face Hub
print(f"Downloading fine-tuned LoRA weights from {HF_LORA_REPO}...")
ADAPTER_PATH = Path(snapshot_download(repo_id=HF_LORA_REPO)).resolve()
assert (ADAPTER_PATH / "adapter_config.json").exists(), "FATAL: adapter_config.json missing from download."
print(f"✓ LoRA weights loaded at: {ADAPTER_PATH}")

# 5. Pre-cache BGE-Base embedding model on CPU
print("Pre-caching BAAI/bge-base-en-v1.5...")
SentenceTransformer("BAAI/bge-base-en-v1.5", device="cpu")
print("✓ All pre-flight checks passed.")

In [ ]:
# CELL 3: LAUNCH vLLM SERVER (PORT 8001)
import os
import sys
import time
import psutil
import subprocess
import requests
from pathlib import Path

# Cleanup port 8001
for proc in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmd = " ".join(proc.info["cmdline"] or [])
        if "vllm" in cmd or "8001" in cmd:
            proc.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass
time.sleep(2)

PROJECT_ROOT = Path("/kaggle/working/ai-gym-coach").resolve()
vllm_log_path = Path("/kaggle/working/vllm.log")
with open(vllm_log_path, "w") as f:
    f.write("--- vLLM Server Startup Log ---\n")

vllm_cmd = [
    sys.executable, "-m", "vllm.entrypoints.openai.api_server",
    "--model", "Qwen/Qwen2.5-7B-Instruct",
    "--host", "0.0.0.0",
    "--port", "8001",
    "--dtype", "half",
    "--quantization", "bitsandbytes",
    "--load-format", "bitsandbytes",
    "--gpu-memory-utilization", "0.90",
    "--max-model-len", "32768",
    "--enable-prefix-caching",
    "--enable-lora",
    "--max-lora-rank", "32",
    "--lora-modules", f"gym_adapter={ADAPTER_PATH}",
    "--enforce-eager"
]

print("[3/5] Launching vLLM Engine in background...")
vllm_proc = subprocess.Popen(
    vllm_cmd,
    stdout=open(vllm_log_path, "a"),
    stderr=subprocess.STDOUT,
    cwd=str(PROJECT_ROOT)
)

print("Awaiting model quantization and LoRA binding...")
vllm_ready = False
log_pos = 0

for attempt in range(120):
    if vllm_log_path.exists():
        with open(vllm_log_path, "r") as f:
            f.seek(log_pos)
            new_lines = f.read()
            if new_lines:
                for line in new_lines.strip().split("\n"):
                    if any(k in line for k in ["Loading", "format", "Route:", "Application startup complete", "Avg prompt", "ERROR"]):
                        print(f"  [vLLM] {line}")
                log_pos = f.tell()

    try:
        r = requests.get("http://127.0.0.1:8001/v1/models", timeout=2)
        if r.status_code == 200:
            vllm_ready = True
            models = [m["id"] for m in r.json().get("data", [])]
            print(f"\n✓ [vLLM LIVE] Models loaded: {models}")
            break
    except Exception:
        pass

    if vllm_proc.poll() is not None:
        with open(vllm_log_path, "r") as f:
            print(f.read())
        raise RuntimeError("vLLM failed to start.")

    time.sleep(5)

if not vllm_ready:
    raise TimeoutError("vLLM did not report healthy within 10 minutes.")

In [ ]:
# CELL 4: LAUNCH FASTAPI GATEWAY (PORT 8000)
import os
import sys
import time
import psutil
import subprocess
import requests
from pathlib import Path

# Cleanup port 8000
for proc in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmd = " ".join(proc.info["cmdline"] or [])
        if "backend.server" in cmd or "8000" in cmd:
            proc.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass
time.sleep(2)

PROJECT_ROOT = Path("/kaggle/working/ai-gym-coach").resolve()

env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT)
env["VLLM_BASE_URL"] = "http://127.0.0.1:8001/v1"
env["ADAPTER_NAME"] = "gym_adapter"
env["BASE_MODEL_NAME"] = "Qwen/Qwen2.5-7B-Instruct"
env["ANONYMIZED_TELEMETRY"] = "False"
env["PYTHONUNBUFFERED"] = "1"

fastapi_log_path = Path("/kaggle/working/fastapi.log")
with open(fastapi_log_path, "w") as f:
    f.write("--- FastAPI Startup Log ---\n")

print("[4/5] Launching FastAPI Gateway on port 8000...")
fastapi_proc = subprocess.Popen(
    [sys.executable, "-m", "backend.server"],
    stdout=open(fastapi_log_path, "a"),
    stderr=subprocess.STDOUT,
    cwd=str(PROJECT_ROOT),
    env=env
)

fastapi_ready = False
for _ in range(30):
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            fastapi_ready = True
            data = r.json()
            print("✓ [FastAPI LIVE] Gateway healthy.")
            print(f"  - Indexed Movements: {data.get('catalog_size')}")
            print(f"  - Active Adapter:   {data.get('adapter_model')}")
            break
    except Exception:
        pass

    if fastapi_proc.poll() is not None:
        with open(fastapi_log_path, "r") as f:
            print(f.read())
        raise RuntimeError("FastAPI crashed on startup.")

    time.sleep(2)

if not fastapi_ready:
    raise RuntimeError("FastAPI failed to report ready within 60s.")

In [ ]:
# CELL 5: NGROK SECURE TUNNEL & REAL-TIME LOG STREAMER
import time
import requests
import getpass
from pathlib import Path
from pyngrok import ngrok

# Optional: Set your own reserved domain, or leave blank for a free random URL
RESERVED_DOMAIN = ""

# 1. Resolve Auth Token (checks Kaggle Secrets first, prompts if missing)
try:
    from kaggle_secrets import UserSecretsClient
    NGROK_AUTH_TOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
except Exception:
    NGROK_AUTH_TOKEN = os.getenv("NGROK_AUTHTOKEN", "")

if not NGROK_AUTH_TOKEN:
    NGROK_AUTH_TOKEN = getpass.getpass("Enter your free Ngrok Authtoken (from dashboard.ngrok.com): ")

# 2. Establish Tunnel
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
ngrok.kill()

if RESERVED_DOMAIN.strip():
    tunnel = ngrok.connect(8000, domain=RESERVED_DOMAIN.strip())
else:
    tunnel = ngrok.connect(8000)

time.sleep(2)

# 3. Instant Zero-Latency Ping
try:
    r = requests.get(
        f"{tunnel.public_url}/health",
        headers={"ngrok-skip-browser-warning": "true"},
        timeout=10
    )
    print("✓ Public gateway confirmed live.")
except Exception as e:
    print(f"⚠️ Tunnel health warning: {e}")

print("=" * 75)
print(f"🚀 API ENDPOINT: {tunnel.public_url}")
print(f"👉 Chat Route:   {tunnel.public_url}/api/v1/chat")
print("=" * 75)
print("Streaming logs below (Press Stop in notebook to halt)...
")

# 4. Real-time Stream
fastapi_log = Path("/kaggle/working/fastapi.log")
vllm_log = Path("/kaggle/working/vllm.log")
fastapi_pos = fastapi_log.stat().st_size if fastapi_log.exists() else 0
vllm_pos = vllm_log.stat().st_size if vllm_log.exists() else 0

try:
    while True:
        if fastapi_log.exists():
            with open(fastapi_log, "r", errors="ignore") as f:
                f.seek(fastapi_pos)
                new_text = f.read()
                if new_text:
                    for line in new_text.splitlines():
                        if line.strip():
                            print(f"[FastAPI] {line}")
                    fastapi_pos = f.tell()

        if vllm_log.exists():
            with open(vllm_log, "r", errors="ignore") as f:
                f.seek(vllm_pos)
                new_text = f.read()
                if new_text:
                    for line in new_text.splitlines():
                        if any(k in line for k in ["Avg prompt", "generation throughput", "HTTP", "POST", "ERROR"]):
                            print(f"  [vLLM] {line.strip()}")
                    vllm_pos = f.tell()
        time.sleep(0.5)
except KeyboardInterrupt:
    print("\nStream paused.")